In [141]:
from email.message import EmailMessage
import os
import smtplib
import requests
from openai.types.responses import ResponseTextDeltaEvent
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, SQLiteSession, OpenAIChatCompletionsModel, set_tracing_disabled, RunHooks, ModelSettings
from IPython.display import Markdown, display
import asyncio
from agents.extensions.visualization import draw_graph
from agents import model_settings
from dotenv import load_dotenv
load_dotenv(override=True)

set_tracing_disabled(disabled=True)

# Define custom local logging hooks
class LocalOrchestrationLogger(RunHooks):
    async def on_agent_start(self, context, agent):
        print(f"\n🔄 [AGENT START] Active Agent switched to: {agent.name}")

    async def on_handoff(self, context, from_agent, to_agent):
        print(f"🔀 [HANDOFF] Control passing from '{from_agent.name}' ➡️ '{to_agent.name}'")

    async def on_tool_start(self, context, tool_definition, arguments):
        print(f"🛠️ [TOOL CALL] Running tool: {tool_definition.name} with args: {arguments}")

    async def on_llm_end(self, context, agent, response):
        print(f"🤖 [LLM RESPONSE] {agent.name} generated a response.")

# Run your multi-agent conversation loop
# res = await runner.run(agent=triage_agent, input="I have a billing issue", hooks=LocalOrchestrationLogger())

ollama_client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
local_llm = OpenAIChatCompletionsModel(model="qwen3:8b", openai_client=ollama_client)
temp_settings = ModelSettings(temperature=0.0)
require_tool = ModelSettings(tool_choice="required")
temp_tool_settings = ModelSettings(tool_choice="required", temperature=0.0)

smtp_server = os.getenv("EMAIL_SMTP_SERVER")
mail_app_password = os.getenv("EMAIL_APP_PASSWORD")
email_address = os.getenv("EMAIL_ADDRESS")
USE_EMAIL = True

if smtp_server and mail_app_password and email_address:
    print("Email config is set")
else:
    print("Email config is not set")

def send_email(subject, text_body, html_body):
    print("send_email invoked")
    msg = EmailMessage()
    msg["From"] = email_address
    msg["To"] = email_address
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(smtp_server, 587) as server:
        server.starttls()
        server.login(email_address, mail_app_password)
        server.send_message(msg)
    print("send_email ended")

@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_email(subject, text_body, html_body)
    return "Email sent successfully"


Email config is set


In [142]:
intro= """
You are a sales agent working for ComplAI,
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write short emails.
"""

professional_email_instructions = intro + "Your email style is professional, serious, with gravities and credibility."
humorous_email_instruction = intro + "Your email is witty, engaging and humorous."
executive_email_instructions = intro + "Your email is concise, to the point, in the style of a busy senior executive."

professional_agent = Agent(name="Professional Sales Agent", model=local_llm, instructions=professional_email_instructions)
humorous_agent = Agent(name="Humorous Sales Agent", model=local_llm, instructions=humorous_email_instruction)
executive_agent = Agent(name="Executive Sales Agent", model=local_llm, instructions=executive_email_instructions)

In [143]:
decision = """
You are an email sending agent.

Your responsibilities are:

1. Read all draft email options.
2. Select exactly one email: the best cold sales email.
3. Extract the following fields from the selected email:
   - subject
   - plain text body
   - HTML body
4. Call the tool:
   send_email_tool(
       subject=<subject>,
       text_body=<plain text body>,
       html_body=<HTML body>
   )

You are not allowed to claim that an email has been sent.

The email is considered sent ONLY after calling send_email_tool.

If send_email_tool has not been called, you must not say "Email Sent".

Your next action must be calling send_email_tool.

Rules:
- Calling send_email_tool is REQUIRED.
- Do NOT return the email to the user instead of calling the tool.
- Do NOT ask for confirmation.
- Call send_email_tool exactly once.
- After the tool succeeds, briefly state that the email has been sent.
"""

tools = [send_email_tool]
print(tools)
sales_sender = Agent(name="Sales_Sender", instructions=decision, model=local_llm, tools=tools)

[FunctionTool(name='send_email_tool', description='Send out an email with the given subject and body to all sales prospects', params_json_schema={'properties': {'subject': {'description': 'The subject of the email', 'title': 'Subject', 'type': 'string'}, 'text_body': {'description': 'The body of the email as plain text', 'title': 'Text Body', 'type': 'string'}, 'html_body': {'description': 'The HTML body of the email', 'title': 'Html Body', 'type': 'string'}}, 'required': ['subject', 'text_body', 'html_body'], 'title': 'send_email_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x0000023E4B8F6000>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)]


In [144]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = professional_agent.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool2 = humorous_agent.as_tool(tool_name="sales_email_writer_2", tool_description=description)
tool3 = executive_agent.as_tool(tool_name="sales_email_writer_3", tool_description=description)

tools = [tool1, tool2, tool3]
handoffs = [sales_sender]

In [145]:
instructions = """ 
You are a Sales Manager at ComplAI. You get your sales_writer tools to draft emails, then send them all to a sales sender.
"""

task = """ 
Follow these steps one by one in order and each step is mandatory:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.

2. Get all the drafted emails and then you must Handoff to the sales_sender agent as a mandatory step so it can choose and send the best email
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, handoffs=handoffs, model=local_llm)

In [146]:
result = await Runner.run(sales_manager, task, hooks=LocalOrchestrationLogger())
print(result.final_output)


🔄 [AGENT START] Active Agent switched to: Sales Manager
🤖 [LLM RESPONSE] Sales Manager generated a response.
🛠️ [TOOL CALL] Running tool: Sales Manager with args: FunctionTool(name='sales_email_writer_1', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x0000023E4B8F6BA0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)
🛠️ [TOOL CALL] Running tool: Sales Manager with args: FunctionTool(name='sales_email_writer_2', description='Use 